# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyanshu-Technologies/flyrank-ML-track/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

I confirm Lane 2: Refresh / Content Opportunity Scoring.

Before writing the rule, I will check two signals that the rule will rely on:

1. `days_since_last_update` — staleness is directly related to FlyRank's refresh-oriented flags.
2. `impressions_90d` — observed search visibility is related to the volume / quick-win logic.

For each signal, I will bucket the values and inspect the number of pages (`n`) in each bucket and the observed declining rate. The declining rate is used only to audit whether the signal is directionally useful; it is not used as an input to the rule.

The rule will prioritize pages that have meaningful search visibility, are relatively stale, and have a position where improvement may be worth human review. It will produce one transparent score, one primary reason code, and one action label.

The rule is decision-support: a high score means "review this page earlier", not "this page will definitely recover after a refresh."

Reason codes:
- `stale_visible_page`
- `visible_position_opportunity`
- `general_review`

Signal verdicts will be recorded as CONFIRMED, OPPOSITE, MIXED, or FALSE after inspecting the bucket tables below.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# 1. Create the baseline score
# A page is prioritized when:
# - it is at least 180 days since its last update
# - it has at least 500 impressions in the 90-day window

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Dataset loaded")
print("Rows:", len(df))
print("Columns:", len(df.columns))

stale_flag = (df["days_since_last_update"] >= 180).astype(int)
visible_flag = (df["impressions_90d"] >= 500).astype(int)

df["score"] = (
    df["impressions_90d"]
    * stale_flag
    * visible_flag
)


# 2. Add reason code

df["reason_code"] = np.where(
    df["score"] > 0,
    "stale_visible_page",
    "not_prioritized"
)


# 3. Add action label

df["action"] = np.where(
    df["score"] > 0,
    "refresh_review",
    "monitor"
)


# 4. Build ranked queue

baseline_queue = df[
    [
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d"
    ]
].copy()

baseline_queue = baseline_queue.sort_values(
    by=["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)



# 5. Add rank

baseline_queue["rank"] = baseline_queue.index + 1

baseline_queue = baseline_queue[
    [
        "rank",
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d"
    ]
]

output_dir = Path("../../work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

baseline_queue.to_csv(
    output_path,
    index=False
)

print("Queue written to:", output_path)
print("Total rows:", len(baseline_queue))

display(baseline_queue.head(20))

# 7. Check the result
print("Queue written to:", output_path)
print("Total rows:", len(baseline_queue))
print("\nTop 20 pages:")

display(baseline_queue.head(20))

Dataset loaded
Rows: 30000
Columns: 44
Queue written to: ../../work/outputs/baseline_action_score.csv
Total rows: 30000


,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,refresh_review,194,61678
1,2,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,refresh_review,194,59472
2,3,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,refresh_review,194,25715
3,4,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,refresh_review,193,13299
4,5,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,refresh_review,194,7812
5,6,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,refresh_review,193,7558
6,7,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,refresh_review,194,4590
7,8,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,refresh_review,194,4556
8,9,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,refresh_review,194,4429
9,10,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,refresh_review,193,1697


Queue written to: ../../work/outputs/baseline_action_score.csv
Total rows: 30000

Top 20 pages:


,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,refresh_review,194,61678
1,2,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,refresh_review,194,59472
2,3,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,refresh_review,194,25715
3,4,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,refresh_review,193,13299
4,5,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,refresh_review,194,7812
5,6,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,refresh_review,193,7558
6,7,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,refresh_review,194,4590
7,8,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,refresh_review,194,4556
8,9,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,refresh_review,194,4429
9,10,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,refresh_review,193,1697


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 20 pages produced by the baseline queue.

The rule prioritizes pages because they are both sufficiently stale and have meaningful observed search visibility.

For each page, the review asks:
- What action did the rule assign?
- Why did the page rank highly?
- What evidence could make this recommendation wrong?

The baseline is intentionally conservative in its interpretation: a high score means the page deserves earlier human review, not that a refresh is guaranteed to improve performance.

In [2]:
# Top-20 review

top20 = baseline_queue.head(20).copy()

top20["review"] = (
    "Action: " + top20["action"]
    + ". Why: stale page with meaningful search visibility. "
    + "What could make it wrong: the page may already satisfy search intent "
      "or have a strategic reason not to be refreshed."
)

display(
    top20[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "review"
        ]
    ]
)

,rank,content_id,score,reason_code,action,review
0,1,content_cf56e2e2e282,61678,stale_visible_page,refresh_review,Action: refresh_review. Why: stale page with m...
1,2,content_7368877ea310,59472,stale_visible_page,refresh_review,Action: refresh_review. Why: stale page with m...
2,3,content_1bfaa38ff26c,25715,stale_visible_page,refresh_review,Action: refresh_review. Why: stale page with m...
3,4,content_0a91db491d14,13299,stale_visible_page,refresh_review,Action: refresh_review. Why: stale page with m...
4,5,content_5feee3994adb,7812,stale_visible_page,refresh_review,Action: refresh_review. Why: stale page with m...
5,6,content_c2d929d83eaa,7558,stale_visible_page,refresh_review,Action: refresh_review. Why: stale page with m...
6,7,content_b16bd7307b39,4590,stale_visible_page,refresh_review,Action: refresh_review. Why: stale page with m...
7,8,content_fe16a55cd13d,4556,stale_visible_page,refresh_review,Action: refresh_review. Why: stale page with m...
8,9,content_ecb6215e79fd,4429,stale_visible_page,refresh_review,Action: refresh_review. Why: stale page with m...
9,10,content_928af3e22c80,1697,stale_visible_page,refresh_review,Action: refresh_review. Why: stale page with m...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


### Weak picks

The baseline can produce false positives because it only considers staleness and search visibility.

A page may be old and receive meaningful impressions while still being a poor refresh candidate. For example, it may already satisfy the user's search intent, have strong positioning, or require a different action such as protection or monitoring.

The rule therefore prioritizes pages for review rather than automatically recommending a refresh.

### Leakage check

The score uses only `days_since_last_update` and `impressions_90d`.

I deliberately exclude:
- `trend_direction`
- `trend_pct`
- future-window performance
- any outcome observed after the decision moment.

The observed decline label is not used to construct the score. It may only be used afterward to evaluate how useful the baseline was.

This keeps the baseline honest and gives the Week-5 model a clean benchmark to beat.

In [3]:
# Leakage check

score_features = [
    "days_since_last_update",
    "impressions_90d"
]

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("Score features:", score_features)
print("Excluded label-derived fields:", forbidden_features)

# Confirm forbidden fields are not part of the queue
queue_feature_columns = set(baseline_queue.columns)

assert not any(
    feature in queue_feature_columns
    for feature in forbidden_features
)

# Recalculate the score independently
check_score = (
    (df["days_since_last_update"] >= 180).astype(int)
    * (df["impressions_90d"] >= 500).astype(int)
    * df["impressions_90d"]
)

assert np.array_equal(
    df["score"].values,
    check_score.values
)

print("\nLeakage check: PASSED")
print("The baseline score uses only pre-decision observable signals.")

Score features: ['days_since_last_update', 'impressions_90d']
Excluded label-derived fields: ['trend_direction', 'trend_pct', 'is_declining_label']

Leakage check: PASSED
The baseline score uses only pre-decision observable signals.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.